In [ ]:
!pip install jiwer

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 12.1 MB/s eta 0:00:00


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

import torchaudio

from torch.utils.data import Dataset, DataLoader

from jiwer import wer

In [ ]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", device)

In [ ]:
hf_dataset = load_dataset(
    "openslr/librispeech_asr",
    "clean",
    split="train.100",
    streaming=True
)

data = []

for i, sample in enumerate(hf_dataset):
    data.append(sample)

    if i == 99:
        break

print(
    "Number of samples:",
    len(data)
)

print(
    "\nFirst sample:"
)

print(
    data[0]
)

In [ ]:
def get_waveform(audio):

    audio_samples = audio.get_all_samples()

    waveform = audio_samples.data

    sample_rate = audio_samples.sample_rate

    return waveform, sample_rate

In [ ]:
waveform, sample_rate = get_waveform(
    data[0]["audio"]
)
print(waveform,sample_rate)

In [ ]:
TARGET_SAMPLE_RATE = 16000

N_MELS = 80

N_FFT = 400

HOP_LENGTH = 160

WIN_LENGTH = 400

In [ ]:
mel_transform = torchaudio.transforms.MelSpectrogram(
    sample_rate=TARGET_SAMPLE_RATE,
    n_fft=N_FFT,
    hop_length=HOP_LENGTH,
    win_length=WIN_LENGTH,
    n_mels=N_MELS,
    window_fn=torch.hann_window
)

In [ ]:
def convert_to_mono(waveform):

    if waveform.ndim == 2:

        waveform = waveform.mean(
            dim=0
        )

    return waveform

In [ ]:
def resample_audio(
    waveform,
    sample_rate
):

    if sample_rate != TARGET_SAMPLE_RATE:

        resampler = torchaudio.transforms.Resample(
            orig_freq=sample_rate,
            new_freq=TARGET_SAMPLE_RATE
        )

        waveform = resampler(
            waveform.unsqueeze(0)
        ).squeeze(0)

    return waveform

In [ ]:
def waveform_to_mel(waveform):

    mel = mel_transform(waveform)

    return mel

In [ ]:
def log_mel(mel):

    mel = torch.log(
        mel + 1e-9
    )

    return mel

In [ ]:
def normalize_features(mel):

    mel = (

        mel - mel.mean()

    ) / (

        mel.std() + 1e-5

    )

    return mel

In [ ]:
def preprocess_audio(audio):

    waveform, sample_rate = get_waveform(
        audio
    )

    waveform = convert_to_mono(
        waveform
    )

    waveform = resample_audio(
        waveform,
        sample_rate
    )

    mel = waveform_to_mel(
        waveform
    )

    mel = log_mel(
        mel
    )

    mel = normalize_features(
        mel
    )

    return mel

In [ ]:
features = preprocess_audio(
    data[0]["audio"]
)

print(
    "\nProcessed features:",
    features.shape
)

In [ ]:
vocab = list(
    "abcdefghijklmnopqrstuvwxyz "
)

In [ ]:
vocab += [
    "<PAD>",
    "<SOS>",
    "<EOS>"
]

In [ ]:
char_to_idx = {
    char: idx
    for idx, char in enumerate(vocab)
}

In [ ]:
idx_to_char = {
    idx: char
    for idx, char in enumerate(vocab)
}

In [ ]:
PAD_IDX = char_to_idx[
    "<PAD>"
]

SOS_IDX = char_to_idx[
    "<SOS>"
]

EOS_IDX = char_to_idx[
    "<EOS>"
]

VOCAB_SIZE = len(vocab)

In [ ]:
print(
    "\nVocabulary size:",
    VOCAB_SIZE
)

print(
    "PAD:",
    PAD_IDX
)

print(
    "SOS:",
    SOS_IDX
)

print(
    "EOS:",
    EOS_IDX
)

In [ ]:
def text_to_tensor(text):

    text = text.lower()

    ids = []

    for char in text:

        if char in char_to_idx:
            ids.append(
                char_to_idx[char]
            )

    ids = (
        [SOS_IDX]
        +
        ids
        +
        [EOS_IDX]
    )

    return torch.tensor(
        ids,
        dtype=torch.long
    )

In [ ]:
def text_to_tensor(text):

    text = text.lower()

    ids = []

    for char in text:

        if char in char_to_idx:

            ids.append(
                char_to_idx[char]
            )

    ids = (
        [SOS_IDX]
        +
        ids
        +
        [EOS_IDX]
    )

    return torch.tensor(
        ids,
        dtype=torch.long
    )

In [ ]:
def tensor_to_text(tokens):

    characters = []

    for token in tokens:

        token = int(token)

        if token == EOS_IDX:
            break

        if token in [
            PAD_IDX,
            SOS_IDX
        ]:
            continue

        if token in idx_to_char:

            characters.append(
                idx_to_char[token]
            )

    return "".join(
        characters
    )

In [ ]:
original_text = data[0]["text"]

labels = text_to_tensor(
    original_text
)

In [ ]:
print("\nOriginal:")
print(original_text)

print("\nLabels:")
print(labels)

print("\nDecoded:")
print(tensor_to_text(labels))

In [ ]:
split_index = int(
    0.9 * len(data)
)

In [ ]:
train_data = data[
    :split_index
]

In [ ]:
print(
    "\nTraining samples:",
    len(train_data)
)

print(
    "Validation samples:",
    len(val_data)
)

In [ ]:
class LASDataset(Dataset):

    def __init__(
        self,
        data
    ):

        self.data = data

    def __len__(self):

        return len(
            self.data
        )

    def __getitem__(
        self,
        index
    ):

        sample = self.data[index]


        features = preprocess_audio(

            sample["audio"]

        )


        text = sample["text"]


        labels = text_to_tensor(
            text
        )

        return (

            features,

            labels,

            text

        )


train_dataset = LASDataset(
    train_data
)

val_dataset = LASDataset(
    val_data
)


In [ ]:
train_dataset = LASDataset(
    train_data
)

val_dataset = LASDataset(
    val_data
)

In [ ]:
def collate_fn(batch):

    features = [
        item[0]
        for item in batch
    ]

    labels = [
        item[1]
        for item in batch
    ]

    texts = [
        item[2]
        for item in batch
    ]

    max_feature_length = max(
        feature.shape[1]
        for feature in features
    )

    padded_features = torch.zeros(
        len(features),
        N_MELS,
        max_feature_length
    )

    feature_lengths = []

    for i, feature in enumerate(
        features
    ):

        length = feature.shape[1]

        padded_features[
            i,
            :,
            :length
        ] = feature

        feature_lengths.append(
            length
        )

    feature_lengths = torch.tensor(
        feature_lengths,
        dtype=torch.long
    )

    max_label_length = max(
        label.shape[0]
        for label in labels
    )

    padded_labels = torch.full(
        (
            len(labels),
            max_label_length
        ),
        PAD_IDX,
        dtype=torch.long
    )

    label_lengths = []

    for i, label in enumerate(
        labels
    ):

        length = label.shape[0]

        padded_labels[
            i,
            :length
        ] = label

        label_lengths.append(
            length
        )

    label_lengths = torch.tensor(
        label_lengths,
        dtype=torch.long
    )

    return (
        padded_features,
        feature_lengths,
        padded_labels,
        label_lengths,
        texts
    )

In [ ]:
BATCH_SIZE = 4

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=collate_fn
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_fn
)

In [ ]:
features, \
feature_lengths, \
labels, \
label_lengths, \
texts = next(
    iter(train_loader)
)

print(
    "\nBatch information:"
)

print(
    "Features:",
    features.shape
)

print(
    "Feature lengths:",
    feature_lengths
)

print(
    "Labels:",
    labels.shape
)

print(
    "Label lengths:",
    label_lengths
)

print(
    "Texts:",
    texts
)

In [ ]:
cnn = nn.Sequential(

    nn.Conv2d(
        in_channels=1,
        out_channels=32,
        kernel_size=3,
        stride=2,
        padding=1
    ),

    nn.BatchNorm2d(32),

    nn.ReLU(),

    nn.Conv2d(
        in_channels=32,
        out_channels=64,
        kernel_size=3,
        stride=2,
        padding=1
    ),

    nn.BatchNorm2d(64),

    nn.ReLU()

).to(device)

In [ ]:
with torch.no_grad():

    x = features.unsqueeze(1)

    x = x.to(device)

    x = cnn(x)

    print(
        "\nCNN output:",
        x.shape
    )

    x = x.permute(
        0,
        2,
        1,
        3
    )

    B, T, C, Freq = x.shape

    CNN_FEATURE_SIZE = C * Freq

print(
    "CNN feature size:",
    CNN_FEATURE_SIZE
)

In [ ]:
class LASEncoder(nn.Module):

    def __init__(
        self,
        input_size,
        hidden_size=256,
        num_layers=2
    ):

        super().__init__()

        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=True
        )

    def forward(self, x):

        outputs, states = self.lstm(x)

        return (
            outputs,
            states
        )

In [ ]:
class Attention(nn.Module):

    def __init__(
        self,
        encoder_dim=512,
        decoder_dim=256
    ):

        super().__init__()

        self.encoder_projection = nn.Linear(
            encoder_dim,
            decoder_dim
        )

        self.decoder_projection = nn.Linear(
            decoder_dim,
            decoder_dim
        )

        self.energy = nn.Linear(
            decoder_dim,
            1
        )

    def forward(
        self,
        encoder_outputs,
        decoder_hidden
    ):

        encoder_features = \
            self.encoder_projection(
                encoder_outputs
            )

        decoder_features = \
            self.decoder_projection(
                decoder_hidden
            )

        decoder_features = \
            decoder_features.unsqueeze(1)

        energy = torch.tanh(
            encoder_features
            +
            decoder_features
        )

        energy = self.energy(
            energy
        )

        energy = energy.squeeze(-1)

        attention_weights = F.softmax(
            energy,
            dim=1
        )

        context = torch.bmm(
            attention_weights.unsqueeze(1),
            encoder_outputs
        )

        context = context.squeeze(1)

        return (
            context,
            attention_weights
        )

In [ ]:
class LASDecoder(nn.Module):

    def __init__(
        self,
        vocab_size,
        embedding_dim=128,
        encoder_dim=512,
        hidden_size=256
    ):

        super().__init__()

        self.embedding = nn.Embedding(
            vocab_size,
            embedding_dim,
            padding_idx=PAD_IDX
        )

        self.attention = Attention(
            encoder_dim,
            hidden_size
        )

        self.lstm = nn.LSTMCell(
            embedding_dim + encoder_dim,
            hidden_size
        )

        self.fc = nn.Linear(
            hidden_size + encoder_dim,
            vocab_size
        )

    def forward_step(
        self,
        input_token,
        hidden,
        cell,
        encoder_outputs
    ):

        embedded = self.embedding(
            input_token
        )

        context, attention_weights = \
            self.attention(
                encoder_outputs,
                hidden
            )

        lstm_input = torch.cat(
            [
                embedded,
                context
            ],
            dim=1
        )

        hidden, cell = self.lstm(
            lstm_input,
            (
                hidden,
                cell
            )
        )

        output = self.fc(
            torch.cat(
                [
                    hidden,
                    context
                ],
                dim=1
            )
        )

        return (
            output,
            hidden,
            cell,
            attention_weights
        )

In [ ]:
encoder = LASEncoder(
    input_size=CNN_FEATURE_SIZE,
    hidden_size=256,
    num_layers=2
).to(device)

decoder = LASDecoder(
    vocab_size=VOCAB_SIZE,
    embedding_dim=128,
    encoder_dim=512,
    hidden_size=256
).to(device)

In [ ]:
optimizer = torch.optim.Adam(
    list(cnn.parameters())
    +
    list(encoder.parameters())
    +
    list(decoder.parameters()),
    lr=1e-3
)

In [ ]:
def train_las_batch(
    features,
    labels
):

    features = features.to(device)

    labels = labels.to(device)

    optimizer.zero_grad()

    x = features.unsqueeze(1)

    x = cnn(x)

    x = x.permute(
        0,
        3,
        1,
        2
    )

    B, T, C, Freq = x.shape

    x = x.reshape(
        B,
        T,
        C * Freq
    )

    encoder_outputs, _ = encoder(x)

    hidden = torch.zeros(
        B,
        256,
        device=device
    )

    cell = torch.zeros(
        B,
        256,
        device=device
    )

    input_token = torch.full(
        (B,),
        SOS_IDX,
        dtype=torch.long,
        device=device
    )

    outputs = []

    max_target_length = labels.shape[1]

    for t in range(
        max_target_length
    ):

        output, \
        hidden, \
        cell, \
        attention_weights = \
            decoder.forward_step(
                input_token,
                hidden,
                cell,
                encoder_outputs
            )

        outputs.append(
            output
        )

        input_token = labels[:, t]

    outputs = torch.stack(
        outputs,
        dim=1
    )

    loss = F.cross_entropy(
        outputs.reshape(
            -1,
            VOCAB_SIZE
        ),
        labels.reshape(-1),
        ignore_index=PAD_IDX
    )

    loss.backward()

    optimizer.step()

    return loss.item()

In [ ]:
NUM_EPOCHS = 20

print(
    "\nStarting LAS training..."
)

for epoch in range(
    NUM_EPOCHS
):

    cnn.train()

    encoder.train()

    decoder.train()

    total_loss = 0

    for batch in train_loader:

        features, \
        feature_lengths, \
        labels, \
        label_lengths, \
        texts = batch

        loss = train_las_batch(
            features,
            labels
        )

        total_loss += loss

    average_loss = (
        total_loss
        /
        len(train_loader)
    )

    print(
        f"Epoch "
        f"{epoch + 1}/{NUM_EPOCHS}"
        f" - Loss: "
        f"{average_loss:.4f}"
    )

In [ ]:
def las_decode(
    features,
    max_length=300
):

    cnn.eval()

    encoder.eval()

    decoder.eval()

    features = features.to(device)

    with torch.no_grad():

        x = features.unsqueeze(1)

        x = cnn(x)

        x = x.permute(
            0,
            3,
            1,
            2
        )

        B, T, C, Freq = x.shape

        x = x.reshape(
            B,
            T,
            C * Freq
        )

        encoder_outputs, _ = encoder(x)

        hidden = torch.zeros(
            B,
            256,
            device=device
        )

        cell = torch.zeros(
            B,
            256,
            device=device
        )

        input_token = torch.full(
            (B,),
            SOS_IDX,
            dtype=torch.long,
            device=device
        )

        predictions = [
            []
            for _ in range(B)
        ]

        finished = torch.zeros(
            B,
            dtype=torch.bool,
            device=device
        )

        for _ in range(
            max_length
        ):

            output, \
            hidden, \
            cell, \
            attention_weights = \
                decoder.forward_step(
                    input_token,
                    hidden,
                    cell,
                    encoder_outputs
                )

            next_token = output.argmax(
                dim=1
            )

            for i in range(B):

                if not finished[i]:

                    predictions[i].append(
                        next_token[i].item()
                    )

            finished = (
                finished
                |
                (
                    next_token == EOS_IDX
                )
            )

            input_token = next_token

            if finished.all():

                break

        decoded_texts = []

        for prediction in predictions:

            text = tensor_to_text(
                prediction
            )

            decoded_texts.append(
                text
            )

    return decoded_texts

In [ ]:
print(
    "\n"
    +
    "=" * 70
)

print(
    "LAS PREDICTIONS"
)

print(
    "=" * 70
)

for i in range(
    min(
        5,
        len(predictions)
    )
):

    print(
        "\nTARGET:"
    )

    print(
        texts[i]
    )

    print(
        "\nPREDICTED:"
    )

    print(
        predictions[i]
    )

    print(
        "-" * 70
)

In [ ]:
wer_scores = []

for reference, hypothesis in zip(
    texts,
    predictions
):

    reference = reference.lower()

    hypothesis = hypothesis.lower()

    score = wer(
        reference,
        hypothesis
    )

    wer_scores.append(
        score
    )

average_wer = (
    sum(wer_scores)
    /
    len(wer_scores)
)

print(
    "\nAverage Validation WER:",
    f"{average_wer:.2%}"
)